In [2]:

import numpy as np

def opt_func(value):
    return np.sqrt((value ** 2).sum())


def gen_frogs(frogs, dimension, sigma, mu):
    return sigma * np.random.randn(frogs, dimension) + mu


def sort_frogs(frogs, mplx_no, opt_func):
    fitness = np.array(list(map(opt_func, frogs)))
    sorted_fitness = np.argsort(fitness)

    memeplexes = np.zeros(
        (mplx_no, int(frogs.shape[0] / mplx_no)),
        dtype=int
    )

    for j in range(memeplexes.shape[1]):
        for i in range(mplx_no):
            memeplexes[i, j] = sorted_fitness[i + (mplx_no * j)]

    return memeplexes


def local_search(frogs, memeplex, opt_func, sigma, mu):
    frog_w = frogs[int(memeplex[-1])]
    frog_b = frogs[int(memeplex[0])]
    frog_g = frogs[np.argmin([opt_func(f) for f in frogs])]

    frog_w_new = frog_w + np.random.rand() * (frog_b - frog_w)

    if opt_func(frog_w_new) >= opt_func(frog_w):
        frog_w_new = frog_w + np.random.rand() * (frog_g - frog_w)

    if opt_func(frog_w_new) >= opt_func(frog_w):
        frog_w_new = gen_frogs(
            1, frogs.shape[1], sigma, mu
        )[0]

    frogs[int(memeplex[-1])] = frog_w_new

    return frogs


def shuffle_memeplexes(frogs, memeplexes):
    temp = memeplexes.flatten()
    np.random.shuffle(temp)

    temp = temp.reshape(
        (memeplexes.shape[0], memeplexes.shape[1])
    )

    return temp


def sfla(opt_func, frogs=30, dimension=2, sigma=1, mu=0,
         mplx_no=5, mplx_iters=10, solun_iters=50):

    frogs = gen_frogs(frogs, dimension, sigma, mu)

    memeplexes = sort_frogs(frogs, mplx_no, opt_func)

    best_idx = np.argmin([opt_func(f) for f in frogs])
    best_solun = frogs[best_idx].copy()

    for i in range(solun_iters):

        memeplexes = shuffle_memeplexes(frogs, memeplexes)

        for memeplex in memeplexes:
            for j in range(mplx_iters):
                frogs = local_search(
                    frogs, memeplex, opt_func, sigma, mu
                )

        memeplexes = sort_frogs(frogs, mplx_no, opt_func)

        new_best_idx = np.argmin(
            [opt_func(f) for f in frogs]
        )

        new_best_solun = frogs[new_best_idx].copy()

        if opt_func(new_best_solun) < opt_func(best_solun):
            best_solun = new_best_solun

        if i % 10 == 0 and i > 1:
            print(
                f"iteration {i}: best solution: {best_solun} "
                f"score: {opt_func(best_solun)}"
            )

    return best_solun, frogs, memeplexes.astype(int)


print("Starting shuffled frog leaping algorithm \n")

solun, frogs, memeplexes = sfla(
    opt_func, 100, 2, 1, 0, 5, 25, 50
)

print("Shuffled frog leaping algorithm completed")

print(
    f"\nBest solution: {solun} Score: {opt_func(solun)}"
)


Starting shuffled frog leaping algorithm 

iteration 10: best solution: [0.07426278 0.07483822] score: 0.10543111256480785
iteration 20: best solution: [-0.07755887 -0.00925222] score: 0.07810878522256415
iteration 30: best solution: [-0.07755887 -0.00925222] score: 0.07810878522256415
iteration 40: best solution: [-0.0563785   0.03023738] score: 0.06397526913420391
Shuffled frog leaping algorithm completed

Best solution: [-0.05908592  0.00705931] Score: 0.059506132763340214
